In [ ]:
# ============================================================
# Notebook 07 - Grad-CAM Visualization and Error Analysis
# CS-419 Deep Learning Project - Speech Emotion Recognition
# ============================================================
# This notebook covers:
#   1. Grad-CAM heatmaps on spectrogram inputs
#   2. Most confident correct predictions
#   3. Most confident wrong predictions (failure cases)
#   4. Per-class confusion analysis
#   5. Accuracy vs parameter count plot
# ============================================================

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

sys.path.append("../src")

from evaluate import evaluate_model, plot_per_class_f1, find_failure_cases
from data_loader import EMOTION_LABELS, EMOTION_DISPLAY
from utils import set_seeds
import tensorflow as tf
from tensorflow import keras

set_seeds(42)
os.makedirs("results/plots", exist_ok=True)

# ---- Cell 1: Load best model and test data ----

print("Loading best model...")
best_model = keras.models.load_model("results/models/best_overall.keras")
best_model.summary()

test_data = np.load("results/cache/test_spec.npz")
X_test, y_test = test_data["X"], test_data["y"]

print(f"Test set: {X_test.shape}")

# ---- Cell 2: Grad-CAM implementation ----

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Compute Grad-CAM heatmap for a single input.

    Parameters
    ----------
    img_array         : np.ndarray  shape (1, H, W, C)
    model             : keras.Model
    last_conv_layer_name : str  name of the last conv layer to inspect
    pred_index        : int or None  class to explain (None = predicted class)

    Returns
    -------
    heatmap : np.ndarray  shape (h, w)  normalized to [0, 1]
    """
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.output,
        ]
    )

    with tf.GradientTape() as tape:
        last_conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_out = last_conv_out[0]
    heatmap = last_conv_out @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(img, heatmap, alpha=0.4):
    """Overlay Grad-CAM heatmap on the spectrogram image."""
    from PIL import Image

    heatmap_resized = np.array(
        Image.fromarray(np.uint8(heatmap * 255)).resize(
            (img.shape[1], img.shape[0]), Image.BILINEAR
        )
    ) / 255.0

    colormap = cm.jet(heatmap_resized)[:, :, :3]
    img_rgb  = np.stack([img[:, :, 0]] * 3, axis=-1)
    overlay  = (1 - alpha) * img_rgb + alpha * colormap
    return np.clip(overlay, 0, 1)


# Find the last conv layer name
last_conv_name = None
for layer in reversed(best_model.layers):
    if isinstance(layer, keras.layers.Conv2D):
        last_conv_name = layer.name
        break
print(f"Last Conv2D layer: {last_conv_name}")

# ---- Cell 3: Grad-CAM on sample spectrograms (one per class) ----

y_prob = best_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

emotions = sorted(set(y_test))
n_emotions = len(emotions)

fig, axes = plt.subplots(n_emotions, 3, figsize=(14, n_emotions * 2.8))
fig.suptitle("Grad-CAM Analysis - What the Model Looks At",
             fontsize=14, fontweight="bold")

col_titles = ["Input Spectrogram", "Grad-CAM Heatmap", "Overlay"]
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=10, fontweight="bold")

for row, emo_idx in enumerate(emotions):
    # Pick a correctly classified sample for this emotion
    correct_mask = (y_test == emo_idx) & (y_pred == emo_idx)
    if not correct_mask.any():
        for ax in axes[row]:
            ax.axis("off")
        continue

    sample_idx = np.where(correct_mask)[0][0]
    img = X_test[sample_idx]  # (H, W, 1)
    img_batch = np.expand_dims(img, axis=0)

    try:
        heatmap = make_gradcam_heatmap(img_batch, best_model, last_conv_name,
                                        pred_index=emo_idx)
        overlay = overlay_gradcam(img, heatmap)
    except Exception as e:
        print(f"Grad-CAM failed for {EMOTION_LABELS[emo_idx]}: {e}")
        for ax in axes[row]:
            ax.axis("off")
        continue

    emo_label = EMOTION_DISPLAY.get(EMOTION_LABELS[emo_idx],
                                     EMOTION_LABELS[emo_idx].title())
    conf = y_prob[sample_idx, emo_idx] * 100

    axes[row, 0].imshow(img[:, :, 0], aspect="auto", origin="lower", cmap="magma")
    axes[row, 0].set_ylabel(f"{emo_label}\n({conf:.1f}%)",
                             fontsize=9, fontweight="bold")

    from PIL import Image
    heatmap_resized = np.array(
        Image.fromarray(np.uint8(heatmap * 255)).resize(
            (img.shape[1], img.shape[0]), Image.BILINEAR
        )
    ) / 255.0
    axes[row, 1].imshow(heatmap_resized, aspect="auto", origin="lower", cmap="jet")
    axes[row, 2].imshow(overlay, aspect="auto", origin="lower")

    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.savefig("results/plots/gradcam_per_emotion.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grad-CAM per emotion saved.")

# ---- Cell 4: Failure case analysis ----

find_failure_cases(X_test, y_test, y_pred, y_prob,
                   n=6, save_dir="results/plots")

# ---- Cell 5: Most confident correct predictions ----

correct_mask = y_test == y_pred
correct_idx  = np.where(correct_mask)[0]
correct_confs = y_prob[correct_idx, y_pred[correct_idx]]
top_correct   = correct_idx[np.argsort(correct_confs)[::-1][:6]]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
fig.suptitle("Most Confident Correct Predictions",
             fontsize=12, fontweight="bold")
axes = axes.flatten()

for i, idx in enumerate(top_correct):
    spec = X_test[idx, :, :, 0]
    true_lbl = EMOTION_DISPLAY.get(EMOTION_LABELS[y_test[idx]],
                                    EMOTION_LABELS[y_test[idx]].title())
    conf = y_prob[idx, y_pred[idx]] * 100
    axes[i].imshow(spec, aspect="auto", origin="lower", cmap="magma")
    axes[i].set_title(f"{true_lbl} ({conf:.1f}%)", fontsize=9, color="green",
                       fontweight="bold")
    axes[i].axis("off")

plt.tight_layout()
plt.savefig("results/plots/correct_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

# ---- Cell 6: Accuracy vs model complexity ----

import pandas as pd

model_stats = [
    {"model": "MLP",           "params": 96*256 + 256*128 + 128*64 + 64*7,  "accuracy": None},
    {"model": "CNN (4 block)", "params": None, "accuracy": None},
    {"model": "CNN-BiLSTM",    "params": None, "accuracy": None},
]

# Load from results CSV
df_results = pd.read_csv("results/all_model_results.csv")

# Get param counts from saved models
mlp_model  = keras.models.load_model("results/models/mlp_best.keras")
cnn_model  = keras.models.load_model("results/models/cnn_best.keras")
best_model2 = keras.models.load_model("results/models/best_overall.keras")

param_data = [
    {"model": "MLP",        "params": mlp_model.count_params(),   "acc_pct": df_results[df_results["model"].str.contains("MLP", na=False)]["accuracy"].max() * 100},
    {"model": "CNN",        "params": cnn_model.count_params(),   "acc_pct": df_results[df_results["model"].str.contains("CNN", na=False) & ~df_results["model"].str.contains("LSTM|GRU|Attn", na=False)]["accuracy"].max() * 100},
    {"model": "CNN-BiLSTM", "params": best_model2.count_params(), "acc_pct": df_results[df_results["model"].str.contains("LSTM|BiLSTM", na=False)]["accuracy"].max() * 100},
]

fig, ax = plt.subplots(figsize=(8, 5))
for row in param_data:
    ax.scatter(row["params"] / 1e6, row["acc_pct"], s=120, zorder=5)
    ax.annotate(row["model"], (row["params"] / 1e6, row["acc_pct"]),
                textcoords="offset points", xytext=(8, 4), fontsize=10)

ax.set_xlabel("Model Parameters (millions)")
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Accuracy vs Model Complexity", fontsize=12, fontweight="bold")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/plots/accuracy_vs_complexity.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== Error Analysis Complete ===")
print("All Grad-CAM and analysis plots saved to results/plots/")